In [21]:
import pandas as pd
import numpy as np
import os

# ======================
# LOAD DATA
# ======================
df = pd.read_csv(r"E:\bankruptcy-risk-system\data\processed\cleaned_data.csv")

# ======================
# CLEAN COLUMN NAMES
# ======================
df.columns = df.columns.str.encode('ascii', 'ignore').str.decode('ascii')
df.columns = df.columns.str.strip().str.replace(" ", "_").str.replace(".", "").str.replace("/", "_")

print("Columns after cleaning:\n", df.columns)

# ======================
# RENAME COLUMNS
# ======================
df.rename(columns={
    "DebtRsCr": "debt",
    "CF_OperationsRsCr": "cash_flow",
    "ROA_12M%": "roa",
    "ROCE%": "roce"
}, inplace=True)

# ======================
# HANDLE MISSING VALUES
# ======================
df = df.replace([np.inf, -np.inf], np.nan)

num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# ======================
# 🔴 FINANCIAL FEATURES
# ======================
df["financial_risk_score"] = df["debt_to_equity"]

df["cash_flow_stress"] = df["debt"] / (abs(df["cash_flow"]) + 1)

df["liquidity_risk"] = 1 / (df["current_ratio"] + 1)

df["fsi"] = (
    df["current_ratio"] +
    df["roa"] +
    df["roce"] -
    df["debt_to_equity"]
)

# ======================
# 🔵 BUSINESS FEATURES
# ======================
df["cost_pressure"] = 1 / (df["roce"] + 1)

df["operational_inefficiency"] = 1 / (df["roce"] + 1)

df["competitive_pressure"] = df["cost_pressure"] + df["operational_inefficiency"]

df["profitability"] = df["roa"] + df["roce"]

# ======================
# 🟣 MANAGEMENT FEATURES
# ======================
df["management_risk"] = (
    (df["roa"] < df["roa"].median()) &
    (df["cash_flow"] < df["cash_flow"].median())
).astype(int)

# ======================
# 🎯 TARGET (NO LEAKAGE IN MODEL INPUT)
# ======================
df["default_flag"] = (
    (df["cash_flow_stress"] > df["cash_flow_stress"].quantile(0.7)) &
    (df["liquidity_risk"] > df["liquidity_risk"].quantile(0.7)) &
    (df["profitability"] < df["profitability"].quantile(0.3))
).astype(int)
print("Target distribution:")
print(df["default_flag"].value_counts())
# ======================
# FINAL FEATURE SET (LEAK-FREE)
# ======================
final_df = df[[
    "company_name",

    # ONLY derived features (safe)
    #"financial_risk_score",
    "cash_flow_stress",
    "liquidity_risk",
    "fsi",
    "cost_pressure",
    "operational_inefficiency",
    "competitive_pressure",
    "profitability",
    "management_risk",

    # TARGET (must include)
    "default_flag"
]]

# ======================
# SAVE
# ======================
BASE_DIR = r"E:\bankruptcy-risk-system"
os.makedirs(os.path.join(BASE_DIR, "data/processed"), exist_ok=True)

final_df.to_csv(os.path.join(BASE_DIR, "data/processed/final_features.csv"), index=False)

print("✅ final_features.csv created successfully")

# ======================
# CHECK TARGET
# ======================
print("\nTarget Distribution:")
print(final_df["default_flag"].value_counts())

# ======================
# EXTRA CHECKS (IMPORTANT)
# ======================
print("\nAny nulls:\n", final_df.isnull().sum())
print("\nAny infinite values:\n", np.isinf(final_df.select_dtypes(include=np.number)).sum())

Columns after cleaning:
 Index(['SNo', 'company_name', 'CMPRs', 'P_E', 'Mar_CapRsCr', 'Div_Yld%',
       'NP_QtrRsCr', 'Qtr_Profit_Var%', 'Sales_QtrRsCr', 'Qtr_Sales_Var%',
       'ROCE%', 'Ind_PE', 'current_ratio', 'debt_to_equity', 'ROA_12M%',
       'DebtRsCr', 'CF_OperationsRsCr', 'inflation'],
      dtype='str')
Target distribution:
default_flag
0    5005
1     348
Name: count, dtype: int64
✅ final_features.csv created successfully

Target Distribution:
default_flag
0    5005
1     348
Name: count, dtype: int64

Any nulls:
 company_name                0
cash_flow_stress            0
liquidity_risk              0
fsi                         0
cost_pressure               0
operational_inefficiency    0
competitive_pressure        0
profitability               0
management_risk             0
default_flag                0
dtype: int64

Any infinite values:
 cash_flow_stress            0
liquidity_risk              0
fsi                         0
cost_pressure               1
operation

In [13]:
import pandas as pd
import numpy as np
import os

# ======================
# LOAD DATA
# ======================
df = pd.read_csv(r"E:\bankruptcy-risk-system\data\processed\cleaned_data.csv")

# ======================
# CLEAN COLUMN NAMES
# ======================
df.columns = df.columns.str.encode('ascii', 'ignore').str.decode('ascii')
df.columns = df.columns.str.strip().str.replace(" ", "_").str.replace(".", "").str.replace("/", "_")

print("Columns after cleaning:\n", df.columns)

# ======================
# RENAME COLUMNS
# ======================
df.rename(columns={
    "DebtRsCr": "debt",
    "CF_OperationsRsCr": "cash_flow",
    "ROA_12M%": "roa",
    "ROCE%": "roce"
}, inplace=True)

# ======================
# HANDLE MISSING VALUES
# ======================
df = df.replace([np.inf, -np.inf], np.nan)

num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# ======================
# 🔴 FINANCIAL FEATURES
# ======================
df["financial_risk_score"] = df["debt_to_equity"]

df["cash_flow_stress"] = df["debt"] / (abs(df["cash_flow"]) + 1)

df["liquidity_risk"] = 1 / (df["current_ratio"] + 1)

df["fsi"] = (
    df["current_ratio"] +
    df["roa"] +
    df["roce"] -
    df["debt_to_equity"]
)

# ======================
# 🔵 BUSINESS FEATURES
# ======================
df["cost_pressure"] = 1 / (df["roce"] + 1)

df["operational_inefficiency"] = 1 / (df["roce"] + 1)

df["competitive_pressure"] = df["cost_pressure"] + df["operational_inefficiency"]

df["profitability"] = df["roa"] + df["roce"]

# ======================
# 🟣 MANAGEMENT FEATURES
# ======================
df["management_risk"] = (
    (df["roa"] < df["roa"].median()) &
    (df["cash_flow"] < df["cash_flow"].median())
).astype(int)

# ======================
# 🎯 TARGET (NO LEAKAGE IN MODEL INPUT)
# ======================
df["default_flag"] = (
    (df["cash_flow_stress"] > df["cash_flow_stress"].quantile(0.7)) &
    (df["liquidity_risk"] > df["liquidity_risk"].quantile(0.7)) &
    (df["profitability"] < df["profitability"].quantile(0.3))
).astype(int)
print("Target distribution:")
print(df["default_flag"].value_counts())

Columns after cleaning:
 Index(['SNo', 'company_name', 'CMPRs', 'P_E', 'Mar_CapRsCr', 'Div_Yld%',
       'NP_QtrRsCr', 'Qtr_Profit_Var%', 'Sales_QtrRsCr', 'Qtr_Sales_Var%',
       'ROCE%', 'Ind_PE', 'current_ratio', 'debt_to_equity', 'ROA_12M%',
       'DebtRsCr', 'CF_OperationsRsCr', 'inflation'],
      dtype='str')
Target distribution:
default_flag
0    5005
1     348
Name: count, dtype: int64
